**Week-1-Logistics-Strategic-Planning**

In [5]:
!pip install -q plotly

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px


# Load datasets
customers = pd.read_csv("olist_customers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")


# Display basic information
print("Customer dataset loaded successfully")
print("Number of records:", len(customers))
print("Number of columns:", len(customers.columns))

print("\nCustomer dataset preview:")
print(customers.head())

print("\nCustomer dataset columns:")
print(customers.columns.tolist())


# Check data quality
print("\nMissing values:")
print(customers.isnull().sum())

print("\nDuplicate rows:")
print(customers.duplicated().sum())


# Calculate KPIs
total_customer_records = len(customers)

unique_customers = customers["customer_unique_id"].nunique()

number_of_states = customers["customer_state"].nunique()

number_of_cities = customers["customer_city"].nunique()


print("\nLogistics KPIs")
print("Total customer records:", total_customer_records)
print("Unique customers:", unique_customers)
print("Number of states:", number_of_states)
print("Number of cities:", number_of_cities)


# Customer distribution by state
state_distribution = (
    customers["customer_state"]
    .value_counts()
    .reset_index()
)

state_distribution.columns = [
    "state",
    "customer_count"
]

state_distribution["percentage"] = (
    state_distribution["customer_count"]
    / total_customer_records
    * 100
)


print("\nTop 10 states:")
print(state_distribution.head(10))


# Customer distribution by city
city_distribution = (
    customers["customer_city"]
    .value_counts()
    .reset_index()
)

city_distribution.columns = [
    "city",
    "customer_count"
]


print("\nTop 10 cities:")
print(city_distribution.head(10))


# Interactive state chart
state_chart = px.bar(
    state_distribution.head(15),
    x="state",
    y="customer_count",
    hover_data=["percentage"],
    title="Top 15 States by Customer Count",
    labels={
        "state": "State",
        "customer_count": "Customers",
        "percentage": "Percentage (%)"
    }
)

state_chart.update_layout(
    xaxis_title="State",
    yaxis_title="Number of Customers"
)

state_chart.show()


# Interactive city chart
city_chart = px.bar(
    city_distribution.head(20),
    x="city",
    y="customer_count",
    title="Top 20 Cities by Customer Count",
    labels={
        "city": "City",
        "customer_count": "Customers"
    }
)

city_chart.update_layout(
    xaxis_tickangle=-45
)

city_chart.show()


# Prepare geolocation data
geo_coordinates = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)


# Rename customer ZIP column
customers_map = customers.rename(
    columns={
        "customer_zip_code_prefix":
        "geolocation_zip_code_prefix"
    }
)


# Count unique customers per ZIP prefix
customer_locations = (
    customers_map
    .groupby("geolocation_zip_code_prefix")
    .agg(
        customer_count=(
            "customer_unique_id",
            "nunique"
        )
    )
    .reset_index()
)


# Merge customers with geographic coordinates
customer_locations = customer_locations.merge(
    geo_coordinates,
    on="geolocation_zip_code_prefix",
    how="left"
)


# Remove locations without coordinates
customer_locations = customer_locations.dropna(
    subset=[
        "latitude",
        "longitude"
    ]
)


print("\nGeographic records available for mapping:")
print(len(customer_locations))


# Interactive customer geographic map
customer_map = px.scatter_map(
    customer_locations,
    lat="latitude",
    lon="longitude",
    size="customer_count",
    color="customer_count",
    hover_name="geolocation_zip_code_prefix",
    hover_data={
        "customer_count": True,
        "latitude": False,
        "longitude": False
    },
    zoom=3,
    height=700,
    title="Interactive Customer Geographic Distribution"
)

customer_map.show()


# Create state-level geographic information
state_geo = (
    customers_map
    .merge(
        geo_coordinates,
        on="geolocation_zip_code_prefix",
        how="left"
    )
)


state_summary = (
    state_geo
    .groupby("customer_state")
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        customer_count=(
            "customer_unique_id",
            "nunique"
        )
    )
    .reset_index()
)


state_summary = state_summary.dropna(
    subset=[
        "latitude",
        "longitude"
    ]
)


# Interactive state map
state_map = px.scatter_geo(
    state_summary,
    lat="latitude",
    lon="longitude",
    size="customer_count",
    color="customer_count",
    hover_name="customer_state",
    hover_data={
        "customer_count": True,
        "latitude": False,
        "longitude": False
    },
    projection="natural earth",
    height=700,
    title="Interactive Customer Distribution by State"
)

state_map.show()


# Identify the largest customer region
top_state = state_distribution.iloc[0]["state"]

top_state_customers = state_distribution.iloc[0][
    "customer_count"
]

top_state_percentage = state_distribution.iloc[0][
    "percentage"
]


print("\nCustomer concentration")
print("Top customer state:", top_state)
print("Customers:", top_state_customers)
print(
    "Percentage:",
    round(top_state_percentage, 2),
    "%"
)


# Strategic insights
print("\nStrategic insights")

print(
    "Customer geographic data can help identify "
    "high-demand regions."
)

print(
    "High customer concentration can support "
    "future warehouse and distribution planning."
)

print(
    "City-level analysis can help identify "
    "important delivery markets."
)

print(
    "Geographic clustering can be used to group "
    "regions according to customer concentration."
)

print(
    "Customer locations can later be combined "
    "with order and delivery data for route optimization."
)


print("\nWeek 1 analysis completed successfully.")

Customer dataset loaded successfully
Number of records: 99441
Number of columns: 5

Customer dataset preview:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

Customer dataset columns:
['customer_id', 


Geographic records available for mapping:
262



Customer concentration
Top customer state: SP
Customers: 41746
Percentage: 41.98 %

Strategic insights
Customer geographic data can help identify high-demand regions.
High customer concentration can support future warehouse and distribution planning.
City-level analysis can help identify important delivery markets.
Geographic clustering can be used to group regions according to customer concentration.
Customer locations can later be combined with order and delivery data for route optimization.

Week 1 analysis completed successfully.
